In [86]:
import pandas as pd
import numpy as np

In [87]:
csv_path = "data/sp500_5yr_with_sectors_weights.csv"

df = pd.read_csv(csv_path)
df.head()

,Date,Open,High,Low,Close,Volume,Symbol,Security,Sector,Weight
0,2021-03-22,131.409549,132.456581,130.537021,132.254166,3115939,MMM,3M,Industrials,0.0013
1,2021-03-23,131.765531,133.817725,131.158259,131.458405,3311365,MMM,3M,Industrials,0.0013
2,2021-03-24,131.528214,134.536678,131.283901,133.105743,3383723,MMM,3M,Industrials,0.0013
3,2021-03-25,133.768854,134.969450,132.742764,134.787964,2656316,MMM,3M,Industrials,0.0013
4,2021-03-26,134.725165,136.079337,134.166757,136.030472,3202410,MMM,3M,Industrials,0.0013


In [88]:
# delete Weight column

df = df.drop(columns=['Weight'])
df.head()


,Date,Open,High,Low,Close,Volume,Symbol,Security,Sector
0,2021-03-22,131.409549,132.456581,130.537021,132.254166,3115939,MMM,3M,Industrials
1,2021-03-23,131.765531,133.817725,131.158259,131.458405,3311365,MMM,3M,Industrials
2,2021-03-24,131.528214,134.536678,131.283901,133.105743,3383723,MMM,3M,Industrials
3,2021-03-25,133.768854,134.969450,132.742764,134.787964,2656316,MMM,3M,Industrials
4,2021-03-26,134.725165,136.079337,134.166757,136.030472,3202410,MMM,3M,Industrials


In [89]:
# make sure date is datetime
df["Date"] = pd.to_datetime(df["Date"])

# sort by stock and date
df = df.sort_values(["Symbol", "Date"]).reset_index(drop=True)

# compute daily log return per stock
df["log_return_1D"] = (
    df.groupby("Symbol")["Close"]
      .transform(lambda s: np.log(s / s.shift(1)))
)

# 1-day absolute return
df["abs_return_1D"] = df["log_return_1D"].abs()

# 1-day squared return
df["sq_return_1D"] = df["log_return_1D"] ** 2

# 5-day rolling volatility
df["roll_vol_5D"] = (
    df.groupby("Symbol")["log_return_1D"]
      .transform(lambda s: s.rolling(5).std())
)

# 21-day rolling volatility
df["roll_vol_21D"] = (
    df.groupby("Symbol")["log_return_1D"]
      .transform(lambda s: s.rolling(21).std())
)

# --- cumulative returns ---
df["cum_return_5D"] = (
    df.groupby("Symbol")["log_return_1D"]
      .transform(lambda s: s.rolling(5).sum())
)

df["cum_return_21D"] = (
    df.groupby("Symbol")["log_return_1D"]
      .transform(lambda s: s.rolling(21).sum())
)

# --- volume features ---
df["log_volume"] = np.log(df["Volume"].replace(0, np.nan))

df["log_volume_change_1D"] = (
    df.groupby("Symbol")["log_volume"]
      .transform(lambda s: s.diff())
)

# target = absolute next-day log return
df["target_nextday_vol"] = (
    df.groupby("Symbol")["log_return_1D"]
      .transform(lambda s: s.shift(-1).abs())
)

In [90]:
df.head(23)

,Date,Open,High,Low,Close,Volume,Symbol,Security,Sector,log_return_1D,abs_return_1D,sq_return_1D,roll_vol_5D,roll_vol_21D,cum_return_5D,cum_return_21D,log_volume,log_volume_change_1D,target_nextday_vol
0,2021-03-22,118.071328,119.675641,117.636428,119.463020,1772900,A,Agilent Technologies,Health Care,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.388127,NaN,0.013274
1,2021-03-23,119.018449,120.062218,117.394811,117.887703,1338300,A,Agilent Technologies,Health Care,-0.013274,0.013274,0.000176,NaN,NaN,NaN,NaN,14.106911,-0.281216,0.009390
2,2021-03-24,118.022998,118.834813,116.689291,116.785934,1477500,A,Agilent Technologies,Health Care,-0.009390,0.009390,0.000088,NaN,NaN,NaN,NaN,14.205862,0.098951,0.008734
3,2021-03-25,116.805284,118.225969,115.858159,117.810394,967300,A,Agilent Technologies,Health Care,0.008734,0.008734,0.000076,NaN,NaN,NaN,NaN,13.782264,-0.423598,0.030219
4,2021-03-26,118.428900,121.492548,117.936008,121.424896,1441900,A,Agilent Technologies,Health Care,0.030219,0.030219,0.000913,NaN,NaN,NaN,NaN,14.181472,0.399208,0.001752
5,2021-03-29,120.690411,121.598877,119.588657,121.212296,1539700,A,Agilent Technologies,Health Care,-0.001752,0.001752,0.000003,0.017426,NaN,0.014537,NaN,14.247098,0.065626,0.004635
6,2021-03-30,120.429477,121.482905,120.236189,120.651756,1033700,A,Agilent Technologies,Health Care,-0.004635,0.004635,0.000021,0.015770,NaN,0.023176,NaN,13.848655,-0.398443,0.018256
7,2021-03-31,121.724516,124.169637,121.724516,122.874596,1815500,A,Agilent Technologies,Health Care,0.018256,0.018256,0.000333,0.014411,NaN,0.050822,NaN,14.411871,0.563216,0.004317
8,2021-04-01,123.705733,123.995671,122.748949,123.406136,1126400,A,Agilent Technologies,Health Care,0.004317,0.004317,0.000019,0.014654,NaN,0.046404,NaN,13.934537,-0.477334,0.019449
9,2021-04-05,124.861889,126.420244,124.861889,125.829811,1096300,A,Agilent Technologies,Health Care,0.019449,0.019449,0.000378,0.011189,NaN,0.035634,NaN,13.907451,-0.027086,0.007968


In [91]:
# drop rows with missing values in features or target

feature_cols = [
    "log_return_1D",
    "abs_return_1D",
    "sq_return_1D",
    "roll_vol_5D",
    "roll_vol_21D",
    "cum_return_5D",
    "cum_return_21D",
    "log_volume_change_1D"
]

target_col = "target_nextday_vol"

df = df.dropna(subset=feature_cols + [target_col]).copy()

In [92]:
# sum of NA values per column

na_counts = df.isna().sum()
print(na_counts)    

Date                    0
Open                    0
High                    0
Low                     0
Close                   0
Volume                  0
Symbol                  0
Security                0
Sector                  0
log_return_1D           0
abs_return_1D           0
sq_return_1D            0
roll_vol_5D             0
roll_vol_21D            0
cum_return_5D           0
cum_return_21D          0
log_volume              0
log_volume_change_1D    0
target_nextday_vol      0
dtype: int64


In [93]:
df.shape
df[feature_cols].describe()

,log_return_1D,abs_return_1D,sq_return_1D,roll_vol_5D,roll_vol_21D,cum_return_5D,cum_return_21D,log_volume_change_1D
count,614542.000000,614542.000000,614542.000000,614542.000000,614542.000000,614542.000000,614542.000000,614542.000000
mean,0.000295,0.013967,0.000439,0.017077,0.018510,0.001469,0.007008,0.000342
std,0.020940,0.015605,0.002276,0.012263,0.009791,0.046034,0.092327,0.387034
min,-0.758010,0.000000,0.000000,0.000000,0.001001,-0.884902,-1.262336,-9.743454
25%,-0.009265,0.004439,0.000020,0.009576,0.012241,-0.021223,-0.043436,-0.229573
50%,0.000623,0.009784,0.000096,0.014082,0.016126,0.002744,0.009171,-0.010848
75%,0.010274,0.018241,0.000333,0.020860,0.021839,0.025537,0.059552,0.218560
max,0.444818,0.758010,0.574579,0.376920,0.180467,0.784955,1.149117,9.656401


In [94]:
# turn df into LSTM sequences per stock

# choose lookback window

LOOKBACK = 21

# create a sequence block by block

import numpy as np

def make_sequences(panel_df, feature_cols, target_col, lookback=30):
    X_list = []
    y_list = []
    meta_list = []

    for symbol, g in panel_df.groupby("Symbol"):
        g = g.sort_values("Date").reset_index(drop=True)

        X_values = g[feature_cols].values
        y_values = g[target_col].values
        dates = g["Date"].values

        for t in range(lookback - 1, len(g)):
            X_window = X_values[t - lookback + 1 : t + 1]
            y_target = y_values[t]

            X_list.append(X_window)
            y_list.append(y_target)
            meta_list.append({
                "Symbol": symbol,
                "Date": dates[t]
            })

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    meta = pd.DataFrame(meta_list)

    return X, y, meta

In [95]:
# run 

X, y, meta = make_sequences(
    panel_df=df,
    feature_cols=feature_cols,
    target_col="target_nextday_vol",
    lookback=30
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("meta shape:", meta.shape)

X shape: (599955, 30, 8)
y shape: (599955,)
meta shape: (599955, 2)


In [96]:
sample_idx = 1500

print(meta.iloc[sample_idx])
print("X sample shape:", X[sample_idx].shape)
print("y sample:", y[sample_idx])
print(pd.DataFrame(X[sample_idx], columns=feature_cols).head())

Symbol                   AAPL
Date      2022-08-03 00:00:00
Name: 1500, dtype: object
X sample shape: (30, 8)
y sample: 0.0019282697
   log_return_1D  abs_return_1D  sq_return_1D  roll_vol_5D  roll_vol_21D  \
0      -0.003834       0.003834      0.000015     0.028036      0.026385   
1       0.021344       0.021344      0.000456     0.028248      0.025287   
2       0.024222       0.024222      0.000587     0.013855      0.025545   
3       0.000000       0.000000      0.000000     0.015826      0.025545   
4      -0.030243       0.030243      0.000915     0.022052      0.025832   

   cum_return_5D  cum_return_21D  log_volume_change_1D  
0       0.019321       -0.016414             -0.098406  
1       0.020754       -0.034405             -0.013376  
2       0.085434        0.009219              0.207275  
3       0.073967        0.008080             -0.238487  
4       0.011489       -0.045097             -0.045524  


In [97]:
all_dates = np.sort(meta["Date"].unique())

train_end = all_dates[int(len(all_dates) * 0.70)]
val_end = all_dates[int(len(all_dates) * 0.85)]

print("train_end:", train_end)
print("val_end:", val_end)

train_end: 2024-10-08T00:00:00.000000000
val_end: 2025-07-01T00:00:00.000000000


In [98]:
train_mask = meta["Date"] <= train_end
val_mask = (meta["Date"] > train_end) & (meta["Date"] <= val_end)
test_mask = meta["Date"] > val_end

In [99]:
X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

meta_train = meta[train_mask].reset_index(drop=True)
meta_val = meta[val_mask].reset_index(drop=True)
meta_test = meta[test_mask].reset_index(drop=True)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(418821, 30, 8) (418821,)
(90726, 30, 8) (90726,)
(90408, 30, 8) (90408,)


In [100]:
n_features = X_train.shape[2]

train_2d = X_train.reshape(-1, n_features)

feature_means = train_2d.mean(axis=0)
feature_stds = train_2d.std(axis=0) + 1e-8

X_train_scaled = (X_train - feature_means) / feature_stds
X_val_scaled = (X_val - feature_means) / feature_stds
X_test_scaled = (X_test - feature_means) / feature_stds

In [101]:
y_mean = y_train.mean()
y_std = y_train.std() + 1e-8

y_train_scaled = (y_train - y_mean) / y_std
y_val_scaled = (y_val - y_mean) / y_std
y_test_scaled = (y_test - y_mean) / y_std

In [102]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(LOOKBACK, len(feature_cols))),
    layers.LSTM(32),
    layers.Dense(16, activation="relu"),
    layers.Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 32)             │         5,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,793 (22.63 KB)

 Trainable params: 5,793 (22.63 KB)

 Non-trainable params: 0 (0.00 B)

In [103]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )
]

history = model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_data=(X_val_scaled, y_val_scaled),
    epochs=30,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/30
6545/6545 ━━━━━━━━━━━━━━━━━━━━ 55s 8ms/step - loss: 0.8607 - mae: 0.6039 - val_loss: 1.2682 - val_mae: 0.6675
Epoch 2/30
6545/6545 ━━━━━━━━━━━━━━━━━━━━ 52s 8ms/step - loss: 0.8505 - mae: 0.6010 - val_loss: 1.2830 - val_mae: 0.6727
Epoch 3/30
6545/6545 ━━━━━━━━━━━━━━━━━━━━ 51s 8ms/step - loss: 0.8459 - mae: 0.5996 - val_loss: 1.2881 - val_mae: 0.6818
Epoch 4/30
6545/6545 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - loss: 0.8416 - mae: 0.5983 - val_loss: 1.2949 - val_mae: 0.6769
Epoch 5/30
6545/6545 ━━━━━━━━━━━━━━━━━━━━ 59s 9ms/step - loss: 0.8383 - mae: 0.5976 - val_loss: 1.3131 - val_mae: 0.6804
Epoch 6/30
6545/6545 ━━━━━━━━━━━━━━━━━━━━ 68s 10ms/step - loss: 0.8346 - mae: 0.5962 - val_loss: 1.2934 - val_mae: 0.6820


In [104]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

y_pred_scaled = model.predict(X_test_scaled).flatten()
y_pred = y_pred_scaled * y_std + y_mean

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
corr = np.corrcoef(y_test, y_pred)[0, 1]

print("Test RMSE:", rmse)
print("Test MAE:", mae)
print("Test correlation:", corr)

# On average, your predicted next-day volatility proxy is off by about 0.93 percentage points of daily return magnitude.

2826/2826 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step
Test RMSE: 0.015188239079734687
Test MAE: 0.009345670230686665
Test correlation: 0.31258656227185416


In [105]:
print("Mean target:", y_test.mean())
print("Median target:", np.median(y_test))
print("Std target:", y_test.std())

Mean target: 0.01393316
Median target: 0.009686589
Std target: 0.015976068


In [106]:
baseline_pred = X_test[:, -1, feature_cols.index("abs_return_1D")]

from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print("Baseline RMSE:", baseline_rmse)
print("Baseline MAE:", baseline_mae)

corr = np.corrcoef(y_test, y_pred)[0, 1]
print("Test correlation:", corr)

Baseline RMSE: 0.020748168235514727
Baseline MAE: 0.012617495842278004
Test correlation: 0.31258656227185416


In [107]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
print("Test R^2:", r2) # 9% of variance explained by the model

Test R^2: 0.096194326877594
